# Rapport préliminaire — DQN, Double DQN et Layer Normalization sur LunarLander

## Objectif du TP

L'objectif demandé est d'étudier empiriquement :

1. **si Double DQN (DDQN) réduit effectivement le biais de surestimation par rapport à DQN** ;
2. cette comparaison **dans le cas standard puis avec des couches de Layer Normalization** ;
3. **l'impact de la Layer Normalization sur le biais de surestimation** ;
4. **l'impact du biais de surestimation sur la performance**, mesurée par la récompense cumulée ;
5. d'appuyer les conclusions sur des **courbes d'apprentissage** et, si possible, des **tests statistiques**.

Ce notebook constitue un **rapport préliminaire** construit à partir des résultats actuellement obtenus. Les tests statistiques détaillés seront ajoutés dans une étape suivante.

## 1. Dispositif expérimental

Nous comparons quatre configurations :

- **DQN**
- **DQN + LayerNorm**
- **DDQN**
- **DDQN + LayerNorm**

Pour chaque configuration :

- environnement : `LunarLander-v2`
- nombre de pas d'entraînement : **300 000**
- nombre de seeds d'entraînement : **5** (`0,1,2,3,4`)
- nombre d'épisodes d'évaluation par modèle : **20**
- mêmes seeds d'évaluation pour tous les modèles : **0 à 19**
- politique d'évaluation : **greedy**
- facteur d'actualisation : **γ = 0.99**

On obtient donc :

\[
4 \text{ configurations} \times 5 \text{ seeds} = 20 \text{ modèles}
\]

et :

\[
20 \text{ modèles} \times 20 \text{ épisodes} = 400 \text{ épisodes d'évaluation}.
\]

L'unité expérimentale principale pour les comparaisons statistiques reste toutefois **le modèle entraîné / la training seed**, et non les 400 épisodes pris comme observations indépendantes.

## 2. DQN vs Double DQN

Dans DQN, la même estimation issue du réseau cible sert à sélectionner et à évaluer l'action :

\[
y_t = r_t + \gamma \max_a Q_{\text{target}}(s_{t+1}, a).
\]

L'opérateur `max` peut favoriser les actions dont l'estimation contient une erreur positive et ainsi produire un **biais de surestimation**.

Dans Double DQN, on sépare :

\[
a^* = \arg\max_a Q_{\text{online}}(s_{t+1},a)
\]

puis :

\[
y_t = r_t + \gamma Q_{\text{target}}(s_{t+1},a^*).
\]

L'objectif est de réduire le couplage entre sélection et évaluation, et donc de diminuer la surestimation.

## 3. Layer Normalization

La Layer Normalization est ajoutée après les deux couches linéaires cachées :

\[
8 \rightarrow 120 \rightarrow LN \rightarrow ReLU
\rightarrow 84 \rightarrow LN \rightarrow ReLU
\rightarrow 4.
\]

Sans LayerNorm :

\[
8 \rightarrow 120 \rightarrow ReLU
\rightarrow 84 \rightarrow ReLU
\rightarrow 4.
\]

La LayerNorm **ne corrige pas directement l'opérateur max de DQN**. Elle agit sur les activations internes du réseau et peut modifier la stabilité et la dynamique d'apprentissage. Son effet sur le biais doit donc être déterminé empiriquement.

## 4. Estimation empirique du biais de surestimation

Pour chaque état-action effectivement visité pendant l'évaluation, on compare :

\[
Q(s_t,a_t)
\]

au retour Monte Carlo observé à partir du même couple \((s_t,a_t)\) :

\[
G_t = r_t + \gamma r_{t+1} + \gamma^2 r_{t+2} + \cdots
\]

Le biais instantané est défini par :

\[
b_t = Q(s_t,a_t)-G_t.
\]

Ainsi :

- \(b_t>0\) : surestimation ;
- \(b_t<0\) : sous-estimation ;
- \(b_t\approx0\) : estimation proche du retour observé.

Le biais moyen d'un épisode est la moyenne des \(b_t\) le long de cet épisode, puis on calcule un biais moyen par modèle.

### Traitement actuel des épisodes tronqués

LunarLander peut interrompre un épisode au **time limit de 1000 pas**. Dans la version actuelle de l'analyse, ces épisodes ne sont plus exclus : le retour est calculé jusqu'au dernier pas observé.

Pour une trajectoire tronquée, il s'agit donc d'un **retour Monte Carlo à horizon fini jusqu'au cutoff**, et non nécessairement du retour jusqu'à une terminaison naturelle.

Cette décision permet :

- de conserver les mêmes 20 épisodes d'évaluation pour tous les modèles ;
- d'éviter un fort biais de sélection dû à l'exclusion systématique des trajectoires longues ;
- d'obtenir une estimation du biais pour les 20 épisodes de chaque modèle.

Cette approximation devra être explicitement mentionnée comme limitation dans le rapport final.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Résultats agrégés actuellement obtenus
condition_summary = pd.DataFrame({
    "condition": ["DQN", "DQN + LN", "DDQN", "DDQN + LN"],
    "mean_return": [-31.393884, 215.477495, -12.115769, 233.851283],
    "std_return": [45.690240, 16.784846, 107.010055, 38.932858],
    "mean_bias": [7.192772, 14.390919, 16.217107, 2.201918],
    "std_bias": [10.953059, 5.090686, 14.984113, 7.378108],
    "mean_truncated_episodes": [16.6, 2.4, 10.4, 1.6],
})

condition_summary

,condition,mean_return,std_return,mean_bias,std_bias,mean_truncated_episodes
0,DQN,-31.393884,45.690240,7.192772,10.953059,16.6
1,DQN + LN,215.477495,16.784846,14.390919,5.090686,2.4
2,DDQN,-12.115769,107.010055,16.217107,14.984113,10.4
3,DDQN + LN,233.851283,38.932858,2.201918,7.378108,1.6


## 5. Résultats agrégés

Les moyennes ci-dessus sont calculées **entre les 5 modèles indépendamment entraînés** pour chaque condition.

Les principaux résultats sont :

| Condition | Return moyen | Biais moyen |
|---|---:|---:|
| DQN | -31.39 | 7.19 |
| DQN + LN | 215.48 | 14.39 |
| DDQN | -12.12 | 16.22 |
| DDQN + LN | 233.85 | 2.20 |

Deux phénomènes apparaissent déjà :

1. **LayerNorm améliore très fortement la performance** dans DQN et DDQN.
2. L'effet sur le biais dépend du type d'algorithme : le biais augmente pour DQN mais diminue fortement pour DDQN.

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(condition_summary["condition"], condition_summary["mean_return"],
        yerr=condition_summary["std_return"], capsize=5)
plt.ylabel("Mean evaluation return")
plt.title("Performance moyenne par condition\n(erreur = écart-type entre training seeds)")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(condition_summary["condition"], condition_summary["mean_bias"],
        yerr=condition_summary["std_bias"], capsize=5)
plt.axhline(0, linestyle="--")
plt.ylabel("Mean overestimation bias")
plt.title("Biais moyen par condition\n(erreur = écart-type entre training seeds)")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Question principale : DDQN réduit-il le biais par rapport à DQN ?

### Cas standard — sans LayerNorm

Les biais moyens sont :

\[
Bias(DQN)=7.19
\]

\[
Bias(DDQN)=16.22.
\]

Avec les résultats actuels, **nous n'observons donc pas de réduction du biais avec DDQN dans le cas standard**. La moyenne est même supérieure pour DDQN.

Il ne faut pas transformer ce résultat en affirmation générale sur DDQN. La conclusion correcte est limitée à notre dispositif expérimental :

> Avec LunarLander, les hyperparamètres utilisés, 300 000 pas d'entraînement et 5 seeds, les résultats obtenus ne montrent pas de réduction empirique du biais de surestimation de DDQN par rapport à DQN dans le cas sans LayerNorm.

### Avec LayerNorm

Les biais moyens deviennent :

\[
Bias(DQN+LN)=14.39
\]

\[
Bias(DDQN+LN)=2.20.
\]

Ici, on observe au contraire une **forte diminution du biais avec DDQN**.

Cette diminution est également cohérente seed par seed : dans les résultats actuels, DDQN+LN présente un biais inférieur à DQN+LN pour chacune des cinq training seeds.

In [ ]:
model_bias_ln = pd.DataFrame({
    "seed": [0, 1, 2, 3, 4],
    "DQN + LN": [8.624066, 10.323349, 15.111808, 16.517597, 21.377775],
    "DDQN + LN": [5.593001, -1.554526, 12.707466, 1.027808, -6.764158],
})

model_bias_ln["difference_DQN_minus_DDQN"] = (
    model_bias_ln["DQN + LN"] - model_bias_ln["DDQN + LN"]
)

model_bias_ln

In [ ]:
plt.figure(figsize=(8, 5))

for _, row in model_bias_ln.iterrows():
    plt.plot(
        ["DQN + LN", "DDQN + LN"],
        [row["DQN + LN"], row["DDQN + LN"]],
        marker="o",
        alpha=0.8
    )

plt.axhline(0, linestyle="--")
plt.ylabel("Mean bias")
plt.title("Comparaison appariée par training seed : DQN+LN vs DDQN+LN")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

### Conclusion provisoire sur DQN vs DDQN

Les résultats actuels montrent une interaction importante avec LayerNorm :

- **sans LayerNorm** : pas de réduction observée du biais avec DDQN ;
- **avec LayerNorm** : réduction nette du biais avec DDQN.

La présence de LayerNorm semble donc modifier fortement la manière dont DQN et DDQN se comportent vis-à-vis de la surestimation.

## 7. Impact de Layer Normalization sur le biais de surestimation

On peut également lire les résultats dans l'autre direction.

### Pour DQN

\[
7.19 \rightarrow 14.39
\]

Le biais moyen **augmente** lorsque LayerNorm est ajouté.

### Pour DDQN

\[
16.22 \rightarrow 2.20
\]

Le biais moyen **diminue fortement** lorsque LayerNorm est ajouté.

La conclusion provisoire n'est donc pas :

> “LayerNorm réduit le biais.”

Elle est plutôt :

> **L'effet de LayerNorm sur le biais dépend de l'algorithme.**

Cela suggère une **interaction algorithm × normalization** :

- LayerNorm augmente le biais mesuré dans DQN ;
- LayerNorm réduit fortement le biais mesuré dans DDQN.

Ce point devra être testé statistiquement avant d'être présenté comme une conclusion définitive.

## 8. Impact de LayerNorm sur la performance

Les returns moyens sont :

\[
R(DQN)=-31.39
\]

\[
R(DQN+LN)=215.48
\]

et :

\[
R(DDQN)=-12.12
\]

\[
R(DDQN+LN)=233.85.
\]

Dans notre expérience, l'effet de LayerNorm sur la performance est donc très important :

- DQN passe d'une performance faible à une politique largement positive ;
- DDQN présente la même amélioration qualitative ;
- DDQN+LN obtient la meilleure performance moyenne des quatre conditions.

LayerNorm réduit également fortement la dispersion des résultats entre seeds pour DQN :

\[
\sigma_R(DQN)=45.69
\]

contre :

\[
\sigma_R(DQN+LN)=16.78.
\]

Ces résultats suggèrent que LayerNorm améliore fortement la stabilité et l'efficacité de l'apprentissage dans ce setup.

## 9. Impact du biais de surestimation sur la performance

À l'échelle des 20 modèles entraînés, la relation entre biais moyen et return moyen est actuellement :

\[
r_{\mathrm{Pearson}}=-0.323
\]

\[
\rho_{\mathrm{Spearman}}=-0.377.
\]

Les deux coefficients sont négatifs.

Cela suggère que, globalement :

\[
Bias \uparrow \quad \Rightarrow \quad Performance \downarrow
\]

mais la relation reste **modérée à faible**.

On ne peut donc pas conclure que le biais de surestimation explique à lui seul les différences de performance.

Un exemple particulièrement intéressant est DQN+LN :

- performance très élevée ;
- biais moyen supérieur à celui de DQN standard.

Ainsi :

\[
\text{faible biais} \neq \text{nécessairement meilleure politique}
\]

et :

\[
\text{biais plus élevé} \neq \text{nécessairement mauvaise performance}.
\]

Le biais est donc un facteur parmi plusieurs qui influencent la dynamique d'apprentissage.

In [ ]:
# Valeurs individuelles actuellement obtenues pour les 20 modèles.
model_results = pd.DataFrame({
    "condition": (
        ["DQN"] * 5
        + ["DQN + LN"] * 5
        + ["DDQN"] * 5
        + ["DDQN + LN"] * 5
    ),
    "seed": list(range(5)) * 4,
    "mean_return": [
        -13.551350, -71.423287, 29.508206, -19.357352, -82.145637,
        196.153402, 237.325349, 228.150359, 208.107971, 207.650393,
        -68.898444, -55.519423, -78.950824, -34.154737, 176.944584,
        225.425360, 251.585107, 169.439674, 259.109642, 263.696632,
    ],
    "mean_bias": [
        3.864806, 5.983203, 0.029707, -0.139835, 26.225978,
        8.624066, 10.323349, 15.111808, 16.517597, 21.377775,
        7.880884, 4.096188, 29.074505, 35.629386, 4.404572,
        5.593001, -1.554526, 12.707466, 1.027808, -6.764158,
    ],
})

pearson = model_results[["mean_bias", "mean_return"]].corr(method="pearson").iloc[0,1]
spearman = model_results[["mean_bias", "mean_return"]].corr(method="spearman").iloc[0,1]

print(f"Pearson  = {pearson:.4f}")
print(f"Spearman = {spearman:.4f}")

In [ ]:
plt.figure(figsize=(8, 5))

markers = {
    "DQN": "o",
    "DQN + LN": "s",
    "DDQN": "^",
    "DDQN + LN": "D",
}

for condition, subset in model_results.groupby("condition"):
    plt.scatter(
        subset["mean_bias"],
        subset["mean_return"],
        label=condition,
        marker=markers[condition],
        s=70
    )

plt.axvline(0, linestyle="--")
plt.xlabel("Mean overestimation bias")
plt.ylabel("Mean evaluation return")
plt.title("Biais de surestimation vs performance\n(un point = un modèle entraîné)")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## 10. Limites actuelles

### 10.1 Seulement cinq training seeds

Chaque condition ne contient que :

\[
n=5
\]

modèles indépendants.

Les statistiques descriptives sont donc informatives, mais les tests auront une puissance limitée.

### 10.2 Trajectoires tronquées

Les modèles sans LayerNorm atteignent souvent la limite de 1000 pas :

- DQN : **16.6 épisodes tronqués sur 20 en moyenne**
- DDQN : **10.4 / 20**
- DQN+LN : **2.4 / 20**
- DDQN+LN : **1.6 / 20**

Pour éviter d'estimer le biais uniquement sur une petite sous-population d'épisodes terminés, le calcul actuel utilise aussi les trajectoires tronquées en calculant leur retour jusqu'au cutoff.

Cela améliore la comparabilité, mais signifie que certains \(G_t\) sont des retours observés à horizon fini.

### 10.3 Corrélation n'implique pas causalité

Même si le biais et la performance sont négativement corrélés, cela ne démontre pas que le biais cause à lui seul la baisse de performance.

Les deux peuvent dépendre de nombreux autres éléments de la dynamique d'apprentissage.

## 11. Conclusions préliminaires

### Réduction du biais avec DDQN

- **Sans LayerNorm** : nous n'observons pas la réduction attendue du biais ; DDQN présente même un biais moyen supérieur à DQN dans nos résultats actuels.
- **Avec LayerNorm** : DDQN réduit fortement le biais par rapport à DQN, de **14.39 à 2.20** en moyenne.

### Impact de LayerNorm sur le biais

LayerNorm n'a pas un effet uniforme :

- DQN : \(7.19 \rightarrow 14.39\), augmentation du biais ;
- DDQN : \(16.22 \rightarrow 2.20\), forte diminution du biais.

Cela suggère une **interaction entre l'algorithme et la normalisation**.

### Impact sur la performance

LayerNorm améliore très fortement la performance :

- DQN : \(-31.39 \rightarrow 215.48\)
- DDQN : \(-12.12 \rightarrow 233.85\)

DDQN+LN obtient la meilleure performance moyenne.

### Biais vs performance

Les corrélations actuellement observées sont négatives :

\[
r=-0.323,\qquad \rho=-0.377
\]

ce qui indique une tendance selon laquelle les modèles ayant davantage de surestimation obtiennent des performances plus faibles.

Cependant, la relation n'est pas forte : **le biais de surestimation n'explique qu'une partie de la performance**.

### Conclusion générale provisoire

Dans notre expérience, **la Layer Normalization est le facteur ayant l'effet le plus visible sur la performance**, tandis que son effet sur le biais dépend fortement du choix entre DQN et DDQN. L'avantage de DDQN en matière de réduction du biais apparaît clairement **lorsque LayerNorm est utilisé**, mais pas dans le cas standard.

Ces conclusions doivent encore être consolidées par des tests statistiques appariés entre training seeds.

## 12. Étapes restantes pour le rapport final

1. Extraire les **learning curves** des logs TensorBoard des 20 runs.
2. Comparer statistiquement, seed par seed :
   - DQN vs DDQN ;
   - DQN+LN vs DDQN+LN ;
   - DQN vs DQN+LN ;
   - DDQN vs DDQN+LN.
3. Ajouter :
   - différences moyennes ;
   - intervalles de confiance ;
   - tests appariés ;
   - tailles d'effet.
4. Tester statistiquement la relation entre bias et performance.
5. Réduire le tout à un rapport final de **4 pages maximum**.